# Visualization

Visualize the best fitted pdb by powerfit with [MolStar](https://molstar.org/).

The notebook expects you have run the [workflow.ipynb](workflow.ipynb) notebook first to generate some results.

<details>
<summary>This notebook must be executed in a Jupyer Lab environment.</summary>

As the Molstart widget needs to download files from a HTTP server like the Jupyer Lab server.

```shell
uv pip install jupyterlab
uv run jupyter lab
```
</details>

![Screenshot of executing MolStar ipywidget](molstar.png "Screenshot of executing MolStar ipywidget")


In [1]:
from pathlib import Path

import duckdb
from ipymolstar import MolViewSpec
from molviewspec import create_builder

from protein_detective.db import db_path, load_fitted_pdbs, load_powerfit_run

In [2]:
import logging

logging.basicConfig(level=logging.INFO)

In [3]:
session_dir = Path("session1")
database = db_path(session_dir)

session_dir

PosixPath('session1')

In [4]:
with duckdb.connect(database) as conn:
    fitted_pdbs = load_fitted_pdbs(conn)
fitted_pdbs

,powerfit_run_id,pdb_file,fitted_file
0,11,single_chain/A8MT69_pdb4ne6.ent_B2A.pdb,fitted_pdbs/1_11_A8MT69_pdb4ne6.ent_B2A.pdb
1,11,single_chain/A8MT69_pdb4drb.ent_J2A.pdb,fitted_pdbs/2_11_A8MT69_pdb4drb.ent_J2A.pdb
2,11,single_chain/A8MT69_pdb4dra.ent_E2A.pdb,fitted_pdbs/3_11_A8MT69_pdb4dra.ent_E2A.pdb
3,11,single_chain/A8MT69_pdb4e44.ent_B2A.pdb,fitted_pdbs/4_11_A8MT69_pdb4e44.ent_B2A.pdb
4,11,single_chain/A8MT69_pdb7xhn.ent_X2A.pdb,fitted_pdbs/5_11_A8MT69_pdb7xhn.ent_X2A.pdb


In [5]:
# TODO move path manipulations to a module
powerfit_run_id = int(fitted_pdbs.iloc[0].powerfit_run_id)
powerfit_run_id

11

In [6]:
fitted_pdb = session_dir / fitted_pdbs.iloc[0].fitted_file
fitted_pdb

PosixPath('session1/fitted_pdbs/1_11_A8MT69_pdb4ne6.ent_B2A.pdb')

In [7]:
with duckdb.connect(database) as conn:
    options, density_map = load_powerfit_run(powerfit_run_id, conn)
options

PowerfitOptions(target=PosixPath('../../powerfit-tutorial/ribosome-KsgA.map'), resolution=13.0, angle=20.0, laplace=True, core_weighted=False, no_resampling=False, resampling_rate=2.0, no_trimming=False, trimming_cutoff=None, gpu=False, nproc=1)

In [8]:
# Reconstruct path to map
adensity_map = session_dir / "powerfit" / str(powerfit_run_id) / density_map
adensity_map

PosixPath('session1/powerfit/11/ribosome-KsgA.map')

In [9]:
# TODO move builder to visualization module
builder = create_builder()

In [10]:
# Download link from inside JupyterLab
# http://localhost:8888/files/docs/session1/fitted_pdbs/1_A8MT69_pdb4e45.ent_B2A.pdb
builder.download(url=f"/files/docs/{fitted_pdb}").parse(
    format="pdb"
).model_structure().component().representation().color(color="blue")

Representation()

In [11]:
builder.download(url=f"/files/docs/{adensity_map}").parse(format="map").volume().representation(
    type="isosurface", relative_isovalue=3, show_wireframe=True
).color(color="green").opacity(opacity=0.1)

VolumeRepresentation()

In [12]:
msvj_data = builder.get_state(indent=2)

In [13]:
view = MolViewSpec(msvj_data=msvj_data.dumps())
view

MolViewSpec(msvj_data='{\n  "kind": "single",\n  "root": {\n    "kind": "root",\n    "children": [\n      {\n …

In [14]:
# TODO nice to have use ipywidgets to select which fitted pdb to draw.